In [1]:
## Importing packages - Please DO NOT alter this box ##
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data.sampler import WeightedRandomSampler
torch.manual_seed(0)

from captum.attr import IntegratedGradients
from captum.attr import DeepLift
from captum.attr import NoiseTunnel
from captum.attr import visualization as viz

import torchvision
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()

import os
import imageio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy.ndimage import binary_erosion
import pandas as pd
from sklearn.metrics import confusion_matrix
import seaborn as sns

import wandb #comment this out if you are not using weights and biases
import random #comment this out if you are not using weights and biases


# Set device to cuda if it's available otherwise default to "cpu"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
images = "/projects/bgmp/shared/Bi625/ML_Assignment/Datasets/Whale_species/species"

In [3]:
transform = transforms.Compose([transforms.ToTensor(), 
                                #We will start with a model called Resnet18 that is optimized for 224x224 images
                                #It is set to a very SMALL size initially so the model will train fast in class
                                transforms.Resize([32,32]),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize the data, these are the values that ResNet suggests based on their training data (natural scences)
                               ])
all_images = datasets.ImageFolder(images, transform)

In [5]:
transform = transforms.Compose([
            transforms.Resize([224,224]), # Resize the image as our model is optimized for 224x224 pixels
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]) # Normalize the data, these are the values that ResNet suggests based on their training data (natural scences))

all_images = datasets.ImageFolder(images, transform )

In [6]:
train_size = int(0.7 * len(all_images))
val_size = int(0.15 * len(all_images))
test_size = len(all_images) - (train_size + val_size)
print(train_size, val_size, test_size)
assert train_size + val_size + test_size == len(all_images)

1955 419 420


In [7]:
train_set, val_set, test_set = torch.utils.data.random_split(all_images, [train_size, val_size, test_size])

In [8]:
def _get_weights(subset,full_dataset):
    ys = np.array([y for _, y in subset])
    counts = np.bincount(ys)
    label_weights = 1.0 / counts
    weights = label_weights[ys]

    print("Number of images per class:")
    for c, n, w in zip(full_dataset.classes, counts, label_weights):
        print(f"\t{c}:\tn={n}\tweight={w}")
        
    return weights

In [11]:
train_weights = _get_weights(train_set,all_images)
train_sampler = WeightedRandomSampler(train_weights, len(train_weights))

Number of images per class:
	beluga:	n=465	weight=0.002150537634408602
	common_dolphin:	n=50	weight=0.02
	false_killer_whale:	n=716	weight=0.0013966480446927375
	fin_whale:	n=116	weight=0.008620689655172414
	gray_whale:	n=44	weight=0.022727272727272728
	humpback_whale:	n=564	weight=0.0017730496453900709


In [17]:
batchsize = 48
learning_rate=1.1e-2
epochs=5

train_loader = DataLoader(train_set, batch_size=batchsize, drop_last=True, sampler=train_sampler)
val_loader = DataLoader(val_set, batch_size=batchsize, drop_last=True, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batchsize, drop_last=True, shuffle=True)

## Model
from torchvision.models import resnet18, ResNet18_Weights

## Getting our model and transferring it to the GPU
#model = ResNet18()

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1, progress  = True).to(device)
nf = model.fc.in_features
model.fc = nn.Linear(nf,6).to(device)

#define loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1.1e-4) # COMMON MOMENTUM, SIMILIAR RELATIVE LR

In [18]:
wandb.init(
    project="BGMP_HappyWhale",
    name="Kadens_resnet18_SGD_V2.0", ##update this with your name
    config={"learning rate":1e-2, # possibly update
        "architecture": "CNN",
        "dataset": "Species",
        "epochs": 5, "batch_size":48}  # possibly update
) 

batch=0
num_epochs = epochs
train_losses, train_acc_list, val_losses, val_acc_list = [], [], [],[]

for epoch in range(num_epochs):
    # Setting model to "training mode"
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    # For each batch of data within the loader
    for inputs, labels in train_loader:
        # Send our input images and their labels to the GPU
        inputs, labels = inputs.to(device), labels.to(device)
        # Zero the gradients
        optimizer.zero_grad()
        # Inputting our training images into the model
        # and Predicting the image classification label
        outputs = model(inputs)
        # Figuring out the loss from our predictions
        loss = criterion(outputs, labels)
        # Compute gradients (aka backward pass)
        loss.backward()
        # Update model parameters
        optimizer.step()

        # Adding the loss to our running sum
        # Loss is calculated for each batch within an epoch
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        batch+=1
        print(f'Epoch [{epoch+1}/{num_epochs}] | Batch #{batch} | Batch Accuracy {(correct/total)*100:.2f}%')


    # Getting metrics for our training pass 
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_acc_list.append(train_acc)
    
    # Switching our model to "evaluation mode"
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0
    # Disable gradient calculation b/c we are evalulating the model
    with torch.no_grad():
        # Load in batches of our validation data
        for inputs, labels in val_loader:
            # Send test images and labels to the GPU
            inputs, labels = inputs.to(device), labels.to(device)
            # Predict the image classification label
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            # Figuring out how many predicted labels = true labels
            correct += predicted.eq(labels).sum().item()

            # Figuring out the loss from our predictions
            loss = criterion(outputs, labels)
            # Adding the loss to our running sum
            # Loss is calculated for each batch within an epoch
            running_loss += loss.item() * inputs.size(0)
    # Getting our accuracy from our test data
    val_acc = 100. * correct / total
    val_acc_list.append(val_acc)
    # Getting the loss from our test data
    val_loss = running_loss / len(val_loader.dataset)
    val_losses.append(val_loss)
    
    print(f'Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss:.2f}%')

    # log metrics to wandb
    wandb.log({"validation_accuracy": val_acc, "validation_loss": val_loss, "train_loss":train_loss})

wandb.finish()

Epoch [1/5] | Batch #1 | Batch Accuracy 22.92%
Epoch [1/5] | Batch #2 | Batch Accuracy 28.12%
Epoch [1/5] | Batch #3 | Batch Accuracy 38.19%
Epoch [1/5] | Batch #4 | Batch Accuracy 43.23%
Epoch [1/5] | Batch #5 | Batch Accuracy 47.08%
Epoch [1/5] | Batch #6 | Batch Accuracy 49.65%
Epoch [1/5] | Batch #7 | Batch Accuracy 55.06%
Epoch [1/5] | Batch #8 | Batch Accuracy 59.38%
Epoch [1/5] | Batch #9 | Batch Accuracy 61.34%
Epoch [1/5] | Batch #10 | Batch Accuracy 62.92%
Epoch [1/5] | Batch #11 | Batch Accuracy 65.53%
Epoch [1/5] | Batch #12 | Batch Accuracy 67.01%
Epoch [1/5] | Batch #13 | Batch Accuracy 68.43%
Epoch [1/5] | Batch #14 | Batch Accuracy 70.09%
Epoch [1/5] | Batch #15 | Batch Accuracy 71.53%
Epoch [1/5] | Batch #16 | Batch Accuracy 73.05%
Epoch [1/5] | Batch #17 | Batch Accuracy 74.02%
Epoch [1/5] | Batch #18 | Batch Accuracy 75.12%
Epoch [1/5] | Batch #19 | Batch Accuracy 76.43%
Epoch [1/5] | Batch #20 | Batch Accuracy 77.19%
Epoch [1/5] | Batch #21 | Batch Accuracy 78.27%
E

train_loss,█▂▁▁▁
validation_accuracy,▆▁█▇█
validation_loss,▂█▁▃▁
train_loss,0.02461
validation_accuracy,96.09375
validation_loss,0.10802
